# Moosic — 03. Cluster Deep-Dive

Investigate one specific final cluster in detail: is it a coherent playlist, or a merge/split artifact? If it's coherent but oversized, test whether it can be cleanly re-split. Starts fresh from `02`'s saved output.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn import set_config

set_config(transform_output="pandas")
RANDOM_STATE = 42

## 2. Load Final Clusters (from Notebook 02)

In [ ]:
df = pd.read_csv("../outputs/songs_final_clusters.csv")
features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

# same scaling philosophy as notebook 02: fit on the FULL dataset, not the subset,
# so distances stay comparable to everything else already computed
scaler = MinMaxScaler().set_output(transform="pandas")
scaled_all = scaler.fit_transform(df[features])

sizes = df['final_cluster'].value_counts()
print(sizes.sort_values(ascending=False).head(10))

## 3. Pick a Cluster to Investigate

In [ ]:
TARGET_CLUSTER = sizes.idxmax()  # defaults to the largest cluster
print(f"Investigating cluster {TARGET_CLUSTER} ({sizes[TARGET_CLUSTER]} songs)")

subset = df[df['final_cluster'] == TARGET_CLUSTER].copy()

### 3a. Experiment — pick a specific cluster ID by hand

Skip Step 3 above and set `TARGET_CLUSTER` directly here if you want to inspect a particular cluster rather than the largest one, e.g.:
```python
TARGET_CLUSTER = 42
subset = df[df['final_cluster'] == TARGET_CLUSTER].copy()
```


## 4. Feature Comparison vs. Dataset Average

Tells you whether this cluster is a **genuinely distinct pocket** of the data (large differences from the overall average) or a **generic catch-all** (close to average on everything — the profile a merge-artifact dumping ground would have).


In [ ]:
overall_mean = df[features].mean()
cluster_mean = subset[features].mean()

comparison = pd.DataFrame({
    "overall_avg": overall_mean,
    "cluster_avg": cluster_mean,
    "difference": cluster_mean - overall_mean
}).round(3)
comparison

## 5. Within-Cluster Spread

Low standard deviation = tight, internally coherent group. High = loose, grab-bag-like — worth treating as a separate warning sign from the average-vs-overall comparison above.


In [ ]:
subset[features].std().round(3)

## 6. Sample Songs — the Real "Does This Feel Like a Playlist" Check

Per the case study's own guidance: metrics tell you about structure, not meaningfulness. This is the step where a human actually has to look.


In [ ]:
sample_size = min(20, len(subset))
print(subset['name'].sample(sample_size, random_state=1).to_string(index=False))

## 7. Re-Split: Explore k=2 to 4

Only worth doing if Steps 4-6 suggested a coherent-but-oversized cluster — re-splitting a genuine grab-bag with no real internal structure won't produce anything cleaner, just smaller arbitrary pieces.


In [ ]:
subset_scaled = scaled_all.loc[subset.index]

print("=== Exploring k=2 to 4 ===")
for k in range(2, 5):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(subset_scaled)
    sil = silhouette_score(subset_scaled, labels)
    sub_sizes = pd.Series(labels).value_counts().sort_index().tolist()
    print(f"k={k}: silhouette={sil:.3f}, sizes={sub_sizes}")

**Rule of thumb applied so far:** pick the k with the *best* (highest) silhouette, not necessarily the smallest or most convenient — a monotonically decreasing silhouette as k increases (like we saw before: 0.413 -> 0.335 -> 0.300) means k=2 is genuinely the best-supported split, not just the simplest.


## 8. Commit to Best k, Inspect Sub-Clusters

In [ ]:
BEST_SUB_K = 2  # set based on Step 7's silhouette results

kmeans = KMeans(n_clusters=BEST_SUB_K, random_state=RANDOM_STATE, n_init=10)
subset['sub_cluster'] = kmeans.fit_predict(subset_scaled)

for sub in sorted(subset['sub_cluster'].unique()):
    group = subset[subset['sub_cluster'] == sub]
    print(f"\n--- Sub-cluster {sub} ({len(group)} songs) ---")
    print("Feature averages:")
    print(group[features].mean().round(3))
    print("\nSample songs:")
    print(group['name'].sample(min(10, len(group)), random_state=1).to_string(index=False))

**Before writing up what separated the groups: check the numbers, not just the song titles.** Song names can suggest a genre/language story that the actual feature averages don't support — worth comparing the two sub-clusters' `Feature averages` directly to see which axis (tempo? energy? valence?) really drove the split, rather than guessing from the sample list alone.


---
**This completes the core Moosic K-Means pipeline (notebooks 01-03).** Next: DBSCAN and Agglomerative Clustering, compared against this baseline on the same lens (size distribution, coherence, business-fit) — the case study's second open question, not yet addressed.
